In [10]:
import numpy as np
import pandas as pd

In [11]:
def near_psd_cov(cov: np.ndarray, eps: float = 0.0) -> np.ndarray:
    cov = 0.5 * (cov + cov.T)

    # Extract standard deviation
    std = np.sqrt(np.diag(cov))
    inverse_std = np.where(std > 0, 1.0 / std, 0.0)

    # Convert covariance to correlation, then force symmetry
    corr = cov * np.outer(inverse_std, inverse_std)
    corr = 0.5 * (corr + corr.T)

    # Eigen decomposition of correlation, and then clip eigenvalues to make PSD
    w, V = np.linalg.eigh(corr)
    w = np.maximum(w, eps)

    # Rebuild PSD correlation
    corr_psd = V @ np.diag(w) @ V.T
    corr_psd = 0.5 * (corr_psd + corr_psd.T)

    # Re-normalize diagonal to 1
    diagonal = np.sqrt(np.diag(corr_psd))
    inverse_diagonal = np.where(diagonal > 0, 1.0 / diagonal, 0.0)
    corr_psd = corr_psd * np.outer(inverse_diagonal, inverse_diagonal)

    # Convert correlation to covariance using original std
    cov_psd = corr_psd * np.outer(std, std)

    return 0.5 * (cov_psd + cov_psd.T)

np.random.seed(0)
cov_input = pd.read_csv("/Users/fuyuxuan/Downloads/test5_3.csv")

# Convert it to numpy array
cov_matrix = cov_input.values

# Apply the function fix to make covariance PSD
cov_fixed = near_psd_cov(cov_matrix, eps=0.0)

# Zero mean vector
mu = np.zeros(cov_fixed.shape[0])

# Simulate 100000 samples
X = np.random.multivariate_normal(mu, cov_fixed, size=100000)

# Compute output covariance matrix
cov_output = np.cov(X, rowvar=False, ddof=0)

# Convert it to DataFrame
cov_output_df = pd.DataFrame(cov_output, columns=cov_input.columns, index=cov_input.columns)

print(cov_output_df)

          x1        x2        x3        x4        x5
x1  0.084891  0.008682  0.037751  0.008052  0.003453
x2  0.008682  0.159457  0.051712  0.010963  0.004733
x3  0.037751  0.051712  0.037348  0.005996  0.002575
x4  0.008052  0.010963  0.005996  0.001686  0.000548
x5  0.003453  0.004733  0.002575  0.000548  0.000313
